# Titanic Dataset — Exploratory Data Analysis

This notebook performs the required exploratory analysis for the Titanic dataset.

**Required analyses covered:**
- Dataset loading and offline fallback
- Shape, information, descriptive statistics, and missingness
- IQR outlier analysis for Age and Fare
- Fare skewness and mean/median/mode comparison
- Survival rates by sex, passenger class, and sex + class
- Exact six-column correlation analysis
- Correlation heatmap and strongest relationships
- Multivariate visualizations with written interpretations


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")

CSV_PATH = "analytics/titanic.csv"

# Load Titanic once from Seaborn when available; use the committed CSV offline fallback.
try:
    titanic = sns.load_dataset("titanic")
    titanic.to_csv(CSV_PATH, index=False)
    print("Loaded Titanic from Seaborn and refreshed the offline CSV fallback.")
except Exception as exc:
    titanic = pd.read_csv(CSV_PATH)
    print(f"Seaborn download unavailable; loaded offline fallback: {CSV_PATH}")
    print(f"Reason: {exc}")

print("Shape:", titanic.shape)
print("\nColumns:")
print(titanic.columns.tolist())


## 1. Dataset profile

The following cells document the dataset shape, data types, summary statistics, and missing values.

In [ ]:
print("SHAPE")
print(titanic.shape)

print("\nINFO")
titanic.info()

print("\nDESCRIPTIVE STATISTICS")
display(titanic.describe(include="all").T)


In [ ]:
missing = pd.DataFrame({
    "missing_count": titanic.isna().sum(),
    "missing_percentage": titanic.isna().mean() * 100
})

missing = missing[missing["missing_count"] > 0].sort_values(
    "missing_percentage", ascending=False
)

print("Missing-value summary:")
display(missing)


## 2. Outlier analysis using the IQR rule

For Age and Fare, the IQR rule defines:
- Lower bound = Q1 − 1.5 × IQR
- Upper bound = Q3 + 1.5 × IQR

Values outside these bounds are counted as outliers.

In [ ]:
def iqr_outlier_summary(series):
    series = series.dropna()
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    count = int(((series < lower) | (series > upper)).sum())
    return {
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "lower_bound": lower,
        "upper_bound": upper,
        "outlier_count": count
    }

outlier_summary = pd.DataFrame({
    "Age": iqr_outlier_summary(titanic["age"]),
    "Fare": iqr_outlier_summary(titanic["fare"])
}).T

display(outlier_summary)


### Outlier interpretation

The IQR counts identify unusually low/high observations without assuming a normal distribution. Fare is expected to show more pronounced high-end outliers because ticket prices are strongly right-skewed, while Age generally has fewer extreme observations.

## 3. Fare distribution and skewness

We compare the mean, median, mode, and skewness of Fare. A mean substantially above the median is evidence of a right-skewed distribution, where a smaller number of expensive tickets pull the average upward.

In [ ]:
fare = titanic["fare"].dropna()

fare_mean = fare.mean()
fare_median = fare.median()
fare_mode = fare.mode().iloc[0]
fare_skew = fare.skew()

fare_summary = pd.Series({
    "mean": fare_mean,
    "median": fare_median,
    "mode": fare_mode,
    "skewness": fare_skew
})

display(fare_summary.to_frame("Fare"))

print(
    f"Interpretation: mean={fare_mean:.2f}, median={fare_median:.2f}, "
    f"mode={fare_mode:.2f}, skewness={fare_skew:.2f}."
)

if fare_skew > 0:
    print("Fare is positively (right) skewed, so high fares pull the mean upward.")
elif fare_skew < 0:
    print("Fare is negatively (left) skewed.")
else:
    print("Fare has approximately symmetric skewness.")


## 4. Survival rates

Survival is examined separately by sex, passenger class, and the combined sex + passenger-class groups.

In [ ]:
sex_survival = titanic.groupby("sex", observed=True)["survived"].mean().mul(100)
pclass_survival = titanic.groupby("pclass", observed=True)["survived"].mean().mul(100)
sex_pclass_survival = (
    titanic.groupby(["sex", "pclass"], observed=True)["survived"]
    .mean()
    .mul(100)
    .reset_index(name="survival_rate_pct")
)

print("Survival rate by sex (%):")
display(sex_survival.to_frame("survival_rate_pct"))

print("Survival rate by passenger class (%):")
display(pclass_survival.to_frame("survival_rate_pct"))

print("Survival rate by sex + passenger class (%):")
display(sex_pclass_survival)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.barplot(
    data=titanic,
    x="sex",
    y="survived",
    estimator="mean",
    errorbar=None,
    ax=axes[0]
)
axes[0].set_title("Survival Rate by Sex")
axes[0].set_ylabel("Survival rate")
axes[0].set_xlabel("Sex")

sns.barplot(
    data=titanic,
    x="pclass",
    y="survived",
    estimator="mean",
    errorbar=None,
    ax=axes[1]
)
axes[1].set_title("Survival Rate by Passenger Class")
axes[1].set_ylabel("Survival rate")
axes[1].set_xlabel("Passenger class")

plt.tight_layout()
plt.show()


**Interpretation:** Survival differs substantially by sex and passenger class. The combined analysis below shows how these two variables interact rather than treating either variable in isolation.

In [ ]:
plt.figure(figsize=(8, 5))

sns.barplot(
    data=titanic,
    x="pclass",
    y="survived",
    hue="sex",
    estimator="mean",
    errorbar=None
)

plt.title("Survival Rate by Passenger Class and Sex")
plt.ylabel("Survival rate")
plt.xlabel("Passenger class")
plt.tight_layout()
plt.show()


## 5. Exact six-column correlation analysis

The required correlation matrix contains exactly:

`survived`, `pclass`, `age`, `sibsp`, `parch`, `fare`

Rows with missing values in these six variables are excluded only for this correlation calculation.

In [ ]:
corr_columns = ["survived", "pclass", "age", "sibsp", "parch", "fare"]

corr_data = titanic[corr_columns].dropna()
correlation_matrix = corr_data.corr()

print("Correlation matrix columns:")
print(correlation_matrix.columns.tolist())

display(correlation_matrix)


In [ ]:
plt.figure(figsize=(8, 6))

sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    square=True
)

plt.title("Titanic Correlation Heatmap")
plt.tight_layout()
plt.show()


In [ ]:
# Find the two strongest absolute off-diagonal correlations.
corr_pairs = correlation_matrix.where(
    np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool)
).stack()

strongest_two = corr_pairs.abs().sort_values(ascending=False).head(2)

print("Two strongest absolute off-diagonal correlations:")

for pair, abs_value in strongest_two.items():
    actual_value = corr_pairs.loc[pair]
    print(f"{pair[0]} vs {pair[1]}: correlation = {actual_value:.3f}")


### Correlation interpretation

The two strongest absolute off-diagonal relationships are identified directly from the required six-variable matrix above.

Correlation measures linear association, not causation. In particular, a correlation with `survived` should be interpreted as an association with the recorded survival outcome rather than evidence that one variable caused the other.

## 6. Multivariate analysis

The following visualizations examine multiple variables simultaneously. Each visualization is followed by an interpretation.

In [ ]:
# Multivariate chart 1: Age distribution by survival and sex
plt.figure(figsize=(9, 5))

sns.histplot(
    data=titanic,
    x="age",
    hue="survived",
    col=None,
    multiple="stack",
    bins=30
)

plt.title("Age Distribution by Survival Outcome")
plt.xlabel("Age")
plt.ylabel("Passenger count")
plt.tight_layout()
plt.show()


**Interpretation:** This chart combines age and survival outcome. It shows where survivors and non-survivors are concentrated across the age distribution, while also making clear that age alone does not completely separate the two groups.

In [ ]:
# Multivariate chart 2: Fare by class and survival
plt.figure(figsize=(10, 5))

sns.boxplot(
    data=titanic,
    x="pclass",
    y="fare",
    hue="survived"
)

plt.title("Fare Distribution by Class and Survival")
plt.xlabel("Passenger class")
plt.ylabel("Fare")
plt.tight_layout()
plt.show()


**Interpretation:** Passenger class, fare, and survival are examined together. Fare distributions differ strongly across classes, and the survival groups can be compared within each class. The chart also highlights high-fare outliers.

In [ ]:
# Multivariate chart 3: Age, fare, class, and survival
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=titanic,
    x="age",
    y="fare",
    hue="survived",
    style="pclass",
    size="pclass",
    sizes=(40, 140),
    alpha=0.7
)

plt.title("Age vs Fare by Survival and Passenger Class")
plt.xlabel("Age")
plt.ylabel("Fare")
plt.tight_layout()
plt.show()


**Interpretation:** This combines age and fare with both survival outcome and passenger class. Passenger class separates many of the observations, while survival is not determined by a single simple boundary in age-fare space.

In [ ]:
# Multivariate chart 4: Family-size variables and survival
family_plot = titanic.copy()
family_plot["family_size"] = family_plot["sibsp"] + family_plot["parch"] + 1

family_survival = (
    family_plot.groupby(["family_size", "pclass"], observed=True)["survived"]
    .mean()
    .reset_index()
)

plt.figure(figsize=(10, 6))

sns.lineplot(
    data=family_survival,
    x="family_size",
    y="survived",
    hue="pclass",
    marker="o"
)

plt.title("Survival Rate by Family Size and Passenger Class")
plt.xlabel("Family size")
plt.ylabel("Survival rate")
plt.tight_layout()
plt.show()


**Interpretation:** Family size and passenger class are considered jointly. Survival rates vary across family sizes, and the pattern differs by passenger class, showing why multivariate analysis is useful beyond one-variable comparisons.

In [ ]:
# Additional multivariate chart: sex, class, and age
plt.figure(figsize=(10, 6))

sns.violinplot(
    data=titanic,
    x="pclass",
    y="age",
    hue="sex",
    split=True,
    inner="quart"
)

plt.title("Age Distribution by Class and Sex")
plt.xlabel("Passenger class")
plt.ylabel("Age")
plt.tight_layout()
plt.show()


**Interpretation:** This visualization compares age distributions across passenger classes while splitting each class by sex. It helps reveal demographic differences that are hidden when age, class, and sex are examined separately.

## EDA conclusion

The Titanic dataset contains missing values, particularly in Age and Cabin, and Fare is positively skewed with high-value outliers. Survival varies across sex and passenger class, and their combination provides additional structure. The required correlation analysis shows that several passenger characteristics are associated with survival, but correlations should not be interpreted causally. The multivariate charts further demonstrate that survival patterns involve interacting demographic and socioeconomic variables rather than a single predictor.